# 0. WATERec options and experiment findings

Start with `README.md`, `docs/capabilities.md` and `docs/experiments.md`. R and RS are equal-size pretrained variants (26.12M parameters), not a Nano/Tiny size ladder. Price SPACE adds 768 parameters. PARSeq Fast Tiny is maintained in its separate repository.

Our qualified plate profile is RS/512; the detailed-price profile is R/2048. Their checkpoints, training pools and timing scopes differ. A trained 17.9M custom reader now reaches 159/206 raw price exact and 186/206 numeric, but remains one AI-reviewed case below Florence and above the 1 ms runtime target. It is documented research, not a selectable notebook model. Dummy examples establish workflow behavior only.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "waterec_ocr").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the repository")
sys.path.insert(0, str(ROOT))
from waterec_ocr.waterec_config import load_config
from waterec_ocr.waterec_models import default_checkpoint
CONFIG_PATH = os.environ.get("WATEREC_CONFIG", "configs/dummy_alpr.yaml")
config = load_config(CONFIG_PATH)
checkpoint = default_checkpoint(config)
deployment = config["output_dir"] / "deployment"
examples = sorted((config["dataset_root"] / "val").glob("*.png"))
IMAGE_PATH = examples[0] if examples else config["dataset_root"] / "val/example.png"
def run(*args):
    return subprocess.run([sys.executable, "-m", "waterec_ocr.waterec_cli", *args, "--config", CONFIG_PATH], cwd=ROOT, check=True)
print("Profile:", CONFIG_PATH, "variant:", config["variant"], "patch budget:", config["tokens"])


In [ ]:
reference = json.loads((ROOT / "results/reference_metrics.json").read_text())
print(json.dumps(reference["model_options"], indent=2))


Choose `WATEREC_CONFIG=configs/dummy_price.yaml` for fictional mixed-size prices. Real-data starting profiles are `waterec_alpr.yaml`, `waterec_r_alpr.yaml`, `waterec_alpr_1024.yaml`, `waterec_price.yaml`, and `waterec_price_512.yaml`. Changing budget requires separate quality checks and matching exports. Default native labels are at most 24 characters with 25 decoder steps including EOS.

## New compact research results

See [the compact research update](../docs/compact_research.md) and [aggregate evidence](../results/compact_research.json). The image-only token selector uses 2048 input patches and 512 late encoder/decoder memory tokens, 17,913,953 parameters, and image/text-only training. TensorRT measured 1.422 ms RGB-to-text and 708 MiB process GPU usage on RTX 5090, static batch 1. These are not edge-device measurements.

Raw 159/206, numeric 186/206 and AI-assisted numeric 132/144 are different metrics. The original strict numeric tolerance failed; a separate text-output qualification matched all 206 native strings. Custom weights/export are not yet included in the supported workflow. The standard R/RS configs below do not instantiate this model.